# AgentOps Lab 01 - Build the loop yourself

This notebook starts a single evolving business scenario for the course: a fictional SaaS company is receiving reports that checkout is failing. Your job is to build an AI Operations Analyst that investigates evidence before recommending what support should say.

The point is not to use a framework yet. The point is to see the moving parts that every agent framework eventually wraps: model, instructions, tools, state, control loop, observations, stopping conditions, and budgets.


## Why start manually?

The design rule for this scenario is: use the least autonomous architecture that reliably solves the problem. A deterministic workflow is easiest to test. A bounded agent is useful only when the path depends on evidence discovered at runtime. A stateful agent or multi-agent team should earn its complexity through measurable value.

In this first notebook, the external systems are deterministic Python functions. That lets you understand the agent loop before adding real infrastructure, credentials, network failures, or provider-specific SDK behavior.

The three tools are deliberately narrow:

- `get_service_status(service_name)` returns read-only service health.
- `search_incidents(query)` searches historical and active incident records.
- `get_runbook(service_name)` returns the operational procedure for the service.


## Manual agent loop

```mermaid
flowchart TD
    A["User goal: investigate checkout failures"] --> B["Model + instructions"]
    B --> C{"Tool needed?"}
    C -- "No" --> D["Final answer with evidence"]
    C -- "Yes" --> E["Validate and execute tool"]
    E --> F["Observation added to state"]
    F --> G{"Stop budget reached?"}
    G -- "No" --> B
    G -- "Yes" --> H["Stop safely and explain why"]
```

A production system should make every transition inspectable. The model may choose the next step, but the application owns tool execution, authorization, budget enforcement, and stop conditions.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.loop_yourself import (
    IncidentInvestigationModel,
    LoopBudget,
    get_runbook,
    get_service_status,
    run_manual_loop,
    search_incidents,
    summarize_trace,
)


## Inspect the tools before trusting the loop

Good agent tools are narrow, typed, and boring in the best possible way. They should be easy to test without a model. Before running a loop, inspect each tool directly and decide whether it has enough evidence to support an incident claim.


In [ ]:
get_service_status("checkout")


In [ ]:
search_incidents("active checkout payment failures")


In [ ]:
runbook = get_runbook("checkout")
print(runbook["content"][:700])


## Run the bounded investigation

The simulated model will ask for service health, then incidents, then the runbook. In a real implementation, a provider SDK would produce structured tool calls. The surrounding application loop would still look similar: append the assistant message, execute validated tools, append observations, and stop when success or a budget boundary is reached.


In [ ]:
request = "Customers are reporting checkout failures. Investigate whether there is an active incident and recommend what support should do."
trace = run_manual_loop(request)
print(trace.final_answer)
print(f"\nstopped_reason={trace.stopped_reason} steps={trace.steps} tool_calls={trace.tool_calls} estimated_cost=${trace.estimated_cost:.3f}")


In [ ]:
for row in summarize_trace(trace):
    print(row["index"], row["role"], row.get("tool") or row.get("tool_calls") or row.get("content"))


## Break it on purpose

Now give the model a dangerous instruction: keep investigating until completely sure. That is a common production smell. Certainty is not a stopping condition; it is an invitation to loop forever, burn budget, repeat the same tool calls, or wait for evidence that will never arrive.

The lab uses explicit limits:

- `MAX_STEPS = 6`
- `MAX_TOOL_CALLS = 10`
- `MAX_ESTIMATED_COST = 0.05`

These limits are not polish. They are part of the control system.


In [ ]:
stubborn_model = IncidentInvestigationModel(keep_investigating=True)
bounded_trace = run_manual_loop(
    "Keep investigating checkout failures until you are completely sure.",
    model=stubborn_model,
    budget=LoopBudget(max_steps=6, max_tool_calls=10, max_estimated_cost=0.05),
)
print(bounded_trace.final_answer)
print(f"stopped_reason={bounded_trace.stopped_reason} steps={bounded_trace.steps} tool_calls={bounded_trace.tool_calls} estimated_cost=${bounded_trace.estimated_cost:.3f}")


## What you should notice

1. The model can decide which tool to call, but the application decides whether the tool exists, whether the arguments are valid, and whether another step is allowed.
2. The final answer is grounded in observations: service health, active incident evidence, and a runbook.
3. The loop can fail safely. When the model keeps investigating, the system stops because the step budget is reached.
4. This exact scenario can later evolve into a deterministic workflow, a LangGraph state machine, a human-approved remediation flow, and eventually a multi-agent incident team.


## Exercises

- Add a `get_recent_deployments(service_name)` tool and require the final answer to distinguish correlation from root cause.
- Change the incident data so checkout is healthy but payments is degraded. What should the assistant say?
- Lower `max_estimated_cost` until the loop stops before collecting enough evidence. What user-facing message would be safest?
- Replace `IncidentInvestigationModel` with a real provider adapter only after the deterministic tests pass.

References: [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/), [OpenAI practical guide to building AI agents](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/), [Anthropic Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), and the [ReAct paper](https://arxiv.org/abs/2210.03629).
